## Local notebook setup

This notebook is intended to be run **locally** from the project root (so relative paths like `data/processed/...` work).

Data locations used in this notebook:
- Raw inputs (optional here): `data/raw/`
- Processed joined dataset (required): `data/processed/hotels_with_cities-2.parquet`


In [ ]:
# Ensure imports work when running locally.
# Run this notebook from the repo root (the folder that contains `src/` and `data/`).

import sys
from pathlib import Path

PROJECT = Path.cwd()
sys.path.insert(0, str(PROJECT))

import src
print("Imported src from:", Path(src.__file__).resolve())

## Verify local files exist

This step confirms the processed dataset exists locally before we load it.

Raw inputs under `data/raw/` are optional here (only needed if you want to regenerate processed outputs).

In [ ]:
from pathlib import Path

PROCESSED = Path("data/processed/hotels_with_cities-2.parquet")
print("processed exists:", PROCESSED.is_file(), PROCESSED)

RAW_HOTELS = Path("data/raw/hotels.csv")
RAW_WORLD = Path("data/raw/worldcitiespop.csv")
print("raw hotels exists:", RAW_HOTELS.is_file(), RAW_HOTELS)
print("raw world cities exists:", RAW_WORLD.is_file(), RAW_WORLD)

In [ ]:
# Install project requirements (local)
# If you already have dependencies installed, you can skip this.

!python3 -m pip install -q -r requirements.txt

## Data wrangling (already completed)

The data-wrangling pipeline (cleaning + standardization + feature creation + Hotels × World Cities join) has already been run, and its output is saved locally as:
- `data/processed/hotels_with_cities-2.parquet`

That processed dataset is what we will use for the rest of this notebook.

If you ever need to regenerate processed outputs from the raw CSVs, you can run the pipeline command shown next—but it is **not needed** for the normal workflow.

In [ ]:
# Do NOT run for the normal workflow.
# The processed dataset is already available at `data/processed/hotels_with_cities-2.parquet`.

RUN_CLEANING = False

if RUN_CLEANING:
    !python3 scripts/pipeline/run_cleaning.py \
        --output-dir data/processed \
        --format parquet \
        --chunksize 50000 \
        --progress-every 50000
else:
    print("Skipping cleaning pipeline (processed dataset already exists).")

## Use the processed dataset

Now we use the processed dataset created above (`data/processed/hotels_with_cities-2.parquet`) and summarize what it contains.

**This cell outputs:**
- Dataset **size** (rows, columns)
- A **5-row preview** (what one hotel record looks like)
- The **top 12 columns by % missing** (coverage snapshot)
- The **distribution of hotel star ratings** (counts per `hotel_star_rating`)

In [ ]:
from pathlib import Path
import pandas as pd

joined = Path("data/processed/hotels_with_cities-2.parquet")
if not joined.is_file():
    raise FileNotFoundError(
        f"Expected processed Parquet at {joined}. "
        "If you saved it under a different name/location, update the path here."
    )

df = pd.read_parquet(joined)

# Normalize any alias column names to the canonical schema.
try:
    from src.modeling.feature_matrix import normalize_engineered_column_names

    df = normalize_engineered_column_names(df)
except Exception:
    pass

print("processed Parquet:", joined)
print("rows:", len(df), "cols:", df.shape[1])

display(df.head(5))

na_rate = (df.isna().mean() * 100).sort_values(ascending=False)
display(na_rate.head(12).to_frame(name="% missing"))

if "hotel_star_rating" not in df.columns:
    raise KeyError("Expected column 'hotel_star_rating' not found in processed dataset.")

display(df["hotel_star_rating"].value_counts(dropna=False).sort_index().to_frame(name="count"))